# 👁️ Participant Starter: Build your ISU System (Track B)

This notebook provides a complete participant workflow for Track B: configure the environment, inspect the provided scenes and auxiliary channels, run an image-understanding system, and save predictions in the expected JSON format.

Your task is to implement the `isu_system(image)` wrapper. It receives one RGB in-car image and must return a JSON-compatible classification for the requested scene features. The reference implementation supports Qwen2.5-VL-3B-Instruct and the lighter Moondream2 baseline.

See the root `README.md` and `track_b_evaluator.ipynb` for the feature definitions, evaluation metrics, and submission requirements.

## 1. ⚙️ Configure the environment

Install the Track B dependencies before running the notebook:

```bash
pip install -r track_b/requirements.txt
```

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from huggingface_hub.utils import HfHubHTTPError
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM, Qwen2_5_VLForConditionalGeneration

NOTEBOOK_DIR = Path.cwd()
TRACK_B_DIR = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'track_b_evaluator.py').exists() else NOTEBOOK_DIR / 'track_b'
REPO_ROOT = TRACK_B_DIR.parent
DATA_DIR = REPO_ROOT / 'data'
DATASET_ID = 'ISU-Test/isu-challenge-dataset'

if str(TRACK_B_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_B_DIR))

from track_b_evaluator import (
    DEFAULT_IGNORED_FEATURES,
    build_prompt,
    build_vocabulary,
    comparable_features,
    load_dataset,
    parse_json_answer,
    query_model,
)

# Xet's parallel chunk requests can hit HF's 429 rate limit on datasets with many files;
# disabling it falls back to plain HTTP downloads with fewer concurrent requests.
os.environ.setdefault('HF_HUB_DISABLE_XET', '1')


def download_dataset_with_retries(max_attempts=5, initial_wait=30):
    """Retry snapshot_download with exponential backoff on HF 429 rate-limit errors."""
    wait_seconds = initial_wait
    for attempt in range(1, max_attempts + 1):
        try:
            snapshot_download(
                repo_id=DATASET_ID,
                repo_type='dataset',
                local_dir=str(DATA_DIR),
                max_workers=2,
            )
            return
        except HfHubHTTPError as error:
            if error.response is not None and error.response.status_code == 429 and attempt < max_attempts:
                print(f'Rate limited (attempt {attempt}/{max_attempts}); retrying in {wait_seconds}s...')
                time.sleep(wait_seconds)
                wait_seconds *= 2
            else:
                raise


# Download the dataset outside track_b when it is not already available locally.
if not any(DATA_DIR.rglob('*_sim.png')):
    print(f'Downloading {DATASET_ID} to {DATA_DIR}...')
    download_dataset_with_retries()

# Choose 'qwen' for the stronger reference model or 'moondream' for a lighter option.
MODEL_BACKEND = 'qwen'
MODEL_NAME = (
    'Qwen/Qwen2.5-VL-3B-Instruct'
    if MODEL_BACKEND == 'qwen'
    else 'vikhyatk/moondream2'
)
IMAGE_KIND = 'both'
SCENE_LIMIT = None
OUTPUT_PATH = TRACK_B_DIR / 'submissions' / 'track_b_predictions.json'
IGNORED_FEATURES = DEFAULT_IGNORED_FEATURES.copy()
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')

print(f'Starter kit data directory: {DATA_DIR}')
print(f'Model: {MODEL_NAME}')
print(f'Device: {DEVICE}')

Starter kit data directory: /home/ubuntu/testing/ISU-Challenge/Starterkit-ICSE-2027/data
Model: Qwen/Qwen2.5-VL-3B-Instruct
Device: cuda


## 2. 📂 Load the provided data

The notebook evaluates discovered image-label pairs. Set `SCENE_LIMIT` above to a small number for a smoke test. Leave it as `None` to process all scenes.

The provided data are organized by scene stem. Files with the same stem belong to the same synthetic scene:

- `images/<stem>_sim.png` — the synthetic RGB image used as an input to the ISU system.
- `labels/<stem>_label.json` — categorical ground-truth scene labels used by the evaluator.
- `canny/<stem>_canny.png` — an edge map derived from the synthetic image.
- `instance_seg/<stem>_instance_seg.png` — a false-color instance-segmentation map; use `instance_seg/instance_seg_legend.json` to decode its colors.
- `depth/png/<stem>_sim_depth.png` and `depth/exr/<stem>_sim_depth.exr` — depth maps in PNG and EXR formats when available.
- Real target-domain reference images — real in-car photos that are separate from the synthetic image-label pairs.

The RGB image, label, and auxiliary channels for a scene describe the same camera view. Track B's starter interface passes the RGB image to `isu_system(image)` and uses the matching label JSON as the expected answer; the other channels are available as additional data for systems that choose to use them.

### Track B prediction features

The system should predict the following observable features from the image. Return one allowed value for each feature requested by the evaluation prompt. A ground-truth value may be `null` when the corresponding passenger or object is not present; the evaluator handles the applicable feature subset for each scene.

| Category | Feature | Possible values | Description |
| --- | --- | --- | --- |
| Driver | `driver_emotion` | `HAPPY`, `SERIOUS` | Driver facial expression. |
| Driver | `driver_phone` | `NO`, `YES` | Whether the driver is using a phone. |
| Driver | `driver_safety_belt` | `NO`, `YES` | Driver safety-belt status. |
| Passenger occupancy | `passenger_codriver` | `NO`, `YES` | Whether the front-right passenger seat is occupied. |
| Codriver | `passenger_codriver_emotion` | `HAPPY`, `SERIOUS` | Codriver facial expression. |
| Codriver | `codriver_safety_belt` | `NO`, `YES` | Codriver safety-belt status. |
| Passenger occupancy | `passenger_back_seat_left` | `NO`, `YES` | Whether the rear-left passenger seat is occupied. |
| Rear-left passenger | `passenger_rear_left_tshirt_color` | `BLACK`, `WHITE` | Rear-left passenger shirt color. |
| Rear-left passenger | `passenger_rear_left_emotion` | `HAPPY`, `SERIOUS` | Rear-left passenger facial expression. |
| Rear-left passenger | `passenger_rear_left_safety_belt` | `NO`, `YES` | Rear-left passenger safety-belt status. |
| Passenger occupancy | `passenger_back_seat_right` | `NO`, `YES` | Whether the rear-right passenger seat is occupied. |
| Rear-right passenger | `passenger_rear_right_tshirt_color` | `BLACK`, `WHITE` | Rear-right passenger shirt color. |
| Rear-right passenger | `passenger_rear_right_safety_belt` | `NO`, `YES` | Rear-right passenger safety-belt status. |
| Objects | `phone_codriver_seat` | `NO`, `YES` | Whether a phone is present on the codriver seat. |
| Objects | `phone_codriver_seat_color` | `BLACK`, `WHITE` | Phone color. |
| Objects | `colabottle_codriver_seat` | `NO`, `YES` | Whether a cola bottle is present on the codriver seat. |
| Objects | `colacan_codriver_seat` | `NO`, `YES` | Whether a cola can is present on the codriver seat. |
| Objects | `suitcase` | `NO`, `YES` | Whether a suitcase is present. |
| Objects | `suitcase_location` | `CO_DRIVER_SEAT`, `REAR_SEAT` | Suitcase location. |
| Objects | `suitcase_pose` | `UPWARDS`, `FLAT` | Suitcase orientation. |
| Child safety | `baby_seat` | `NO`, `YES` | Whether a baby seat is present. |
| Child safety | `baby_seat_orientation` | `FRONT_FACING`, `REAR_FACING` | Baby-seat orientation. |
| Child safety | `baby` | `NO`, `YES` | Whether a baby is present. |

In [3]:
scenes = load_dataset(DATA_DIR, IMAGE_KIND)
if SCENE_LIMIT is not None:
    scenes = scenes[:SCENE_LIMIT]
vocabulary = build_vocabulary(scenes, IGNORED_FEATURES)
PROMPT = build_prompt(vocabulary)

print(f'Loaded {len(scenes)} image-label pairs')
print('Features:', ', '.join(vocabulary))

Loaded 1 image-label pairs
Features: baby_seat, codriver_safety_belt, colabottle_codriver_seat, colacan_codriver_seat, driver_emotion, driver_gender, driver_phone, driver_safety_belt, driver_tshirt_color, env, passenger_back_seat_left, passenger_back_seat_right, passenger_front_seat_right, passenger_rear_left_emotion, passenger_rear_left_tshirt_color, passenger_rear_right_emotion, passenger_rear_right_tshirt_color, phone_codriver_seat, phone_codriver_seat_color, rear_left_safety_belt, rear_right_safety_belt, suitcase


## 3. 🖼️ Browse scenes and auxiliary channels

Use the controls below to inspect the images selected by `IMAGE_KIND`. The viewer displays the RGB image, matching Canny edge map, instance-segmentation map, depth map when available, and scene-label JSON. Auxiliary files are matched using the base scene stem; unavailable channels are shown as such.

In [ ]:
import ipywidgets as widgets
from IPython.display import HTML, clear_output, display


def auxiliary_paths(scene_name):
    stem = scene_name[:-len('_sim')] if scene_name.endswith('_sim') else scene_name
    return {
        'Canny': DATA_DIR / 'canny' / f'{stem}_canny.png',
        'Instance segmentation': DATA_DIR / 'instance_seg' / f'{stem}_instance_seg.png',
        'Depth': DATA_DIR / 'depth' / 'png' / f'{stem}_sim_depth.png',
    }


def image_widget(path, width=260):
    if not path.exists():
        return widgets.HTML(f'<i>Not available</i><br><small>{path}</small>')
    return widgets.Image(value=path.read_bytes(), format=path.suffix.lstrip('.'), width=width)


def show_scene(scene_index):
    scene_name, image_path, label = scenes[scene_index]
    panels = [
        widgets.VBox([widgets.HTML('<b>RGB image</b>'), image_widget(image_path)]),
    ]
    for title, path in auxiliary_paths(scene_name).items():
        panels.append(widgets.VBox([widgets.HTML(f'<b>{title}</b>'), image_widget(path)]))

    clear_output(wait=True)
    display(widgets.HBox(panels, layout=widgets.Layout(overflow='auto')))
    display(HTML(f'<b>Scene:</b> {scene_name}'))
    display(HTML(f'<pre>{json.dumps(label, indent=2, ensure_ascii=True)}</pre>'))


scene_selector = widgets.Dropdown(
    options=[scene_name for scene_name, _, _ in scenes],
    value=scenes[0][0],
    description='Scene:',
)
previous = widgets.Button(description='Previous', icon='arrow-left')
next_button = widgets.Button(description='Next', icon='arrow-right')
output = widgets.Output()


def render_selected(change=None):
    selected_index = next(index for index, scene in enumerate(scenes) if scene[0] == scene_selector.value)
    with output:
        show_scene(selected_index)


def move_scene(step):
    current_index = next(index for index, scene in enumerate(scenes) if scene[0] == scene_selector.value)
    scene_selector.value = scenes[(current_index + step) % len(scenes)][0]


scene_selector.observe(render_selected, names='value')
previous.on_click(lambda _: move_scene(-1))
next_button.on_click(lambda _: move_scene(1))
display(widgets.HBox([previous, next_button, scene_selector]))
display(output)
render_selected()

## 4. 🤖 Choose a model and implement the SUT wrapper

Set `MODEL_BACKEND` in the setup cell to `qwen` for the reference Qwen2.5-VL-3B-Instruct model or `moondream` for the lighter `vikhyatk/moondream2` model. Moondream requires fewer resources and can run on CPU; Qwen generally provides a stronger baseline but requires substantially more memory and compute. Both options use the same `isu_system(image)` interface.

Implement `isu_system(image)` yourself below. It receives **only one RGB `PIL.Image.Image`** as input and must return a JSON-compatible dictionary with the requested feature names and one classification value for each feature.

The system must not receive or use the ground-truth label, expected feature values, scene metadata, auxiliary channels, or any other contextual information about the scene. Those data are available to the notebook for dataset inspection and evaluation, but they are not inputs to the participant system. The prediction must be based solely on the image provided to `isu_system(image)`.

The marked body is the participant implementation point. The reference body supports both example models and can be replaced by any reproducible model or pipeline.

In [4]:
if MODEL_BACKEND == 'qwen':
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype='auto',
        device_map=DEVICE,
    ).eval()
else:
    processor = None
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        revision='2025-06-21',
        trust_remote_code=True,
    ).to(DEVICE).eval()


def isu_system(image: Image.Image) -> dict:
    """Return one feature classification for one RGB image."""
    # TODO: Participants must implement this wrapper with their own ISU system.
    # This reference implementation supports Qwen and the lighter Moondream option.
    if MODEL_BACKEND == 'qwen':
        raw_answer = query_model(model, processor, image, PROMPT, DEVICE)
    else:
        raw_answer = model.query(
            image,
            PROMPT,
            reasoning=False,
            settings={'temperature': 0.0, 'top_p': 0.0, 'max_tokens': 768},
        )['answer']
    prediction = parse_json_answer(raw_answer)
    if prediction is None:
        raise ValueError('isu_system() must return a valid JSON object.')
    return prediction

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## 5. 🔎 Generate predictions

The evaluator calls the same `isu_system(image)` interface for every selected image. Each output records the image name, prediction, expected labels, feature-level matches, scene-level exact match, and inference time.

In [4]:
prediction_rows = []

for index, (scene_name, image_path, label) in enumerate(scenes, 1):
    image = Image.open(image_path).convert('RGB')
    started = time.perf_counter()
    prediction = isu_system(image)
    expected = comparable_features(label, IGNORED_FEATURES)
    feature_matches = {
        key: str(prediction.get(key)).strip().upper() == str(value).strip().upper()
        for key, value in expected.items()
    }
    prediction_rows.append({
        'image': scene_name,
        'image_path': str(image_path),
        'expected': expected,
        'prediction': prediction,
        'feature_matches': feature_matches,
        'all_features_match': all(feature_matches.values()),
        'inference_seconds': time.perf_counter() - started,
    })
    print(f'[{index}/{len(scenes)}] {scene_name}', flush=True)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[1/1] sample_0001_sim


## 6. 💾 Save predictions

The resulting JSON report includes the selected model, device, evaluated feature vocabulary, per-image predictions, feature matches, exact-match status, and inference times. It can be inspected locally and supplied to the Track B evaluation workflow.

In [5]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
report = {
    'track': 'B',
    'model': MODEL_NAME,
    'device': DEVICE,
    'features': vocabulary,
    'predictions': prediction_rows,
}
OUTPUT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=True))
print(f'Saved predictions to {OUTPUT_PATH}')

Saved predictions to /home/ubuntu/testing/ISU-Challenge/Starterkit-ICSE-2027/track_b/submissions/track_b_predictions.json
